# PathoVision — Colab End-to-End Workflow

This notebook runs the real pipeline. It contains no fabricated outputs. Put BreakHis under the configured `data/raw/` path or mount it from Google Drive.

In [ ]:
!nvidia-smi
!pip -q install -r requirements.txt

In [ ]:
from pathlib import Path
print("Working directory:", Path.cwd())
# If the repository is not already present, clone your GitHub repository here, then cd into it.
# Example after publishing: !git clone <your-repository-url>
# %cd PathoVision

## 1. Dataset path

Upload/clone the repository and place BreakHis under `data/raw/`. The parser discovers the directory structure instead of assuming a single hard-coded layout.

In [ ]:
from pathlib import Path
raw = Path("data/raw")
print("Dataset root exists:", raw.exists())
if not raw.exists():
    raise FileNotFoundError("Place BreakHis under data/raw/ before continuing.")

In [ ]:
!python scripts/prepare_data.py --config configs/default.yaml --validate-images

## 2. Inspect tiling and tissue filtering

In [ ]:
import pandas as pd
from PIL import Image
from src.config import load_config
from src.data.tiling import select_tile_coordinates, extract_tile

cfg = load_config("configs/default.yaml")
meta = pd.read_csv(cfg["paths"]["metadata_csv"])
row = meta.iloc[0]
img = Image.open(row.image_path).convert("RGB")
coords = select_tile_coordinates(img, cfg["dataset"]["tile_size"], cfg["dataset"]["stride"], cfg["dataset"]["tissue_threshold"], cfg["dataset"]["background_gray_threshold"], seed=cfg["seed"])
print("Image:", row.image_path)
print("Image size:", img.size)
print("Valid tiles:", len(coords))
if coords:
    display(extract_tile(img, coords[0].x, coords[0].y, cfg["dataset"]["tile_size"]))

## 3. Run tests

In [ ]:
!pytest -q

## 4. Train

For a fast demonstration, edit the dataset limits/epochs in a copied config using the `fast` values. Keep the default configuration for final experiments.

In [ ]:
!python scripts/train.py --config configs/default.yaml

## 5. Evaluate the untouched test split

In [ ]:
!python scripts/evaluate.py --config configs/default.yaml

In [ ]:
import json
from pathlib import Path
metrics = json.loads(Path("results/metrics/test_metrics.json").read_text())
metrics

## 6. Show generated figures

In [ ]:
from IPython.display import display, Image as IPImage
from pathlib import Path
for p in sorted(Path("results/figures").glob("*.png")):
    print(p)
    display(IPImage(filename=str(p), width=600))

## 7. Single-image inference

Replace `sample.png` with a real BreakHis image path. Grad-CAM is optional.

In [ ]:
# !python scripts/predict.py --config configs/default.yaml --image path/to/sample.png --gradcam

## 8. Generated artifacts

The run produces checkpoint metadata, metrics, predictions, curves, confusion matrix, ROC curve, aggregation comparison, magnification analysis, and optional Grad-CAM. Do not publish fabricated values; only copy values from the generated files.